In [1]:
import os  # for file operations

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    ArrayType,
)
from pyspark.sql.functions import (
    col,
    when,
    sum as spark_sum,
    round as spark_round,
    regexp_replace,
    split,
    udf,
    expr,
    explode,
)

# Setting our constants

In [2]:
ROOT_DIR = ""
PATH_PREFIX = "/mnt/data/public/insideairbnb/data.insideairbnb.com/"

START_DATE = "2019-01-01"
END_DATE = "2025-12-31"

CITY_CONFIGS = [
    {
        "label": "new-york-city",
        "root_path": f"{PATH_PREFIX}united-states/ny/new-york-city",
        "regex_key": "united-states/ny/new-york-city",
    },
    {
        "label": "los-angeles",
        "root_path": f"{PATH_PREFIX}united-states/ca/los-angeles",
        "regex_key": "united-states/ca/los-angeles",
    },
    {
        "label": "chicago",
        "root_path": f"{PATH_PREFIX}united-states/il/chicago",
        "regex_key": "united-states/tx/austin",
    },
]

# Loading the Dataset

The Inside Airbnb dataset is organized by month, so each city has a separate folder for every period. To analyze the data, we first need to gather all the months post-pandemic, which we have to define to be after 2021. For each month, we read `listings.csv` to get the price information and `listings.csv.gz` to get the amenities offered by each listing. The function `build_city_airbnb_df` combines all these months into a single Spark DataFrame for each city and save the combined data as a Parquet file for later use.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os

def load_city_files(city_config, start_year=2021, base_dir=PATH_PREFIX):
    spark = SparkSession.builder.appName(f"Airbnb_{city_config['label']}").getOrCreate()
    
    city_base_dir = os.path.join(base_dir, city_config["regex_key"])

    if not os.path.exists(city_base_dir):
        raise FileNotFoundError(f"Directory not found: {city_base_dir}")

    folders = [
        f for f in os.listdir(city_base_dir)
        if f[:4].isdigit() and int(f[:4]) >= start_year
    ]

    columns_to_keep = [
        "id",
        "price",
        "number_of_reviews",
        "minimum_nights",
        "room_type",
        "latitude",
        "longitude",
        "property_type",
        "accommodates"
    ]
    
    df_data_list = []
    df_viz_list = []
    
    for folder in sorted(folders):
        month_path = os.path.join(city_base_dir, folder)

        data_filepath = os.path.join(month_path, "data/listings.csv.gz")
        viz_filepath = os.path.join(month_path, "visualisations/listings.csv")

        # --- Load DATA file as STRING columns ---
        if os.path.exists(data_filepath):
            df_data = spark.read.csv(data_filepath, header=True, inferSchema=False)
            df_data = df_data.select([col(c).cast("string") for c in columns_to_keep if c in df_data.columns])
            df_data_list.append(df_data)

        # --- Load VIZ file as STRING columns ---
        if os.path.exists(viz_filepath):
            df_viz = spark.read.csv(viz_filepath, header=True, inferSchema=False)
            df_viz = df_viz.select([col(c).cast("string") for c in columns_to_keep if c in df_viz.columns])
            df_viz_list.append(df_viz)

    # --- Combine DATA ---
    df_data_combined = None
    if df_data_list:
        df_data_combined = df_data_list[0]
        for df in df_data_list[1:]:
            df_data_combined = df_data_combined.unionByName(df)

    # --- Combine VIZ ---
    df_viz_combined = None
    if df_viz_list:
        df_viz_combined = df_viz_list[0]
        for df in df_viz_list[1:]:
            df_viz_combined = df_viz_combined.unionByName(df)

    # --- JOIN ---
    if df_data_combined is None and df_viz_combined is None:
        return None

    if df_data_combined is None:
        return df_viz_combined
    
    if df_viz_combined is None:
        return df_data_combined

    overlapping_cols = [c for c in df_viz_combined.columns if c in df_data_combined.columns and c != "id"]
    df_viz_clean = df_viz_combined.drop(*overlapping_cols)

    return df_data_combined.join(df_viz_clean, on="id", how="outer")


In [4]:
for city_config in CITY_CONFIGS:
    print(f"\n===== Processing {city_config['label']} =====")

    try:
        df_city = load_city_files(city_config, start_year=2021)

        if df_city is None:
            print("No data — both DATA and VIZ folders empty.")
            continue

        print(f"Rows: {df_city.count():,}")
        print(f"Columns: {len(df_city.columns)}")
        print(df_city.printSchema())

    except Exception as e:
        print(f"ERROR loading {city_config['label']}: {e}")


===== Processing new-york-city =====
Rows: 5,348,002
Columns: 9
root
 |-- id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: string (nullable = true)

None

===== Processing los-angeles =====
Rows: 4,750,268
Columns: 9
root
 |-- id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: string (nullable = true)

None

===== Processing chicago =====
Rows: 1,214,759
Columns: 9
root
 |-- id: string (nullable = true

# Checking for Null Values

For each city, some listings have missing values, and since these rows are incomplete, they are not suitable for modeling. We therefore remove any listings with null values in either column to ensure a clean and reliable dataset.

After this filtering:
- New York City retains 355,392 listings (67% of the original dataset)
- Los Angeles retains 380,924 listings (73%)
- Chicago retains 103,685 listings (67%)

These remaining datasets are sufficiently large to support robust analysis and predictive modeling.

In [5]:
for city in CITY_CONFIGS:
    print(city['label'].replace('-', ' ').title())
    city_df = load_city_files(city, start_year=2021)
    total = city_df.count()
    remaining = city_df.dropna().count()
    print(f"Initial Size of {total} (100%)")
    print(f"Non-empty rows of {remaining} ({int(round(remaining/total, 2) * 100)}%).")
    total_rows = city_df.count()  # total number of rows
    
    na_counts_percentage = city_df.select(
        [
            (spark_round(
                1-(spark_sum(when(col(c).isNull(), 1).otherwise(0)) / total_rows),
                2
            ) * 100).alias(f'{c} (%)')
            for c in city_df.columns
        ]
    )
    
    na_counts_percentage.show()

New York City
Initial Size of 5348002 (100%)
Non-empty rows of 3783920 (71%).
+------+---------+---------------------+------------------+-------------+------------+-------------+-----------------+----------------+
|id (%)|price (%)|number_of_reviews (%)|minimum_nights (%)|room_type (%)|latitude (%)|longitude (%)|property_type (%)|accommodates (%)|
+------+---------+---------------------+------------------+-------------+------------+-------------+-----------------+----------------+
| 100.0|     75.0|                 72.0|              75.0|         75.0|        75.0|         75.0|             75.0|            75.0|
+------+---------+---------------------+------------------+-------------+------------+-------------+-----------------+----------------+

Los Angeles
Initial Size of 4750268 (100%)
Non-empty rows of 3401974 (72%).
+------+---------+---------------------+------------------+-------------+------------+-------------+-----------------+----------------+
|id (%)|price (%)|number_of_r

# Cleaning the amenities

We preprocessed Airbnb data for each city by first removing any rows with missing values and safely converting numeric columns to floats, dropping rows where conversion failed. Next, we encoded all categorical string columns using one-hot encoding while discarding the original string and intermediate index columns. Finally, the cleaned and transformed datasets were optionally saved as Parquet files for efficient storage and downstream analysis.

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, expr
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
import os

spark = SparkSession.builder.getOrCreate()

def preprocess_city(city_config, save_parquet=True, output_dir="data/preprocessed"):
    """
    Preprocess city Airbnb data:
    1. Drop rows with any NULLs.
    2. Safely convert numeric columns to float (malformed entries become NULL then dropped).
    3. One-hot encode string/categorical columns.
    """
    df = load_city_files(city_config)

    # --- Drop rows with any NULLs ---
    df = df.dropna(how='any')

    # --- List numeric columns ---
    numeric_cols = ["price", "number_of_reviews", "minimum_nights", "latitude", "longitude", "accommodates"]

    # --- Safely cast numeric columns ---
    for c in numeric_cols:
        if c in df.columns:
            # Remove quotes and extra spaces, then try cast
            df = df.withColumn(c, expr(f"try_cast(regexp_replace({c}, '\"', '') as float)"))

    # --- Drop any rows where numeric cast failed ---
    df = df.dropna(subset=numeric_cols)

    # --- Identify categorical/string columns ---
    string_cols = ['room_type', 'property_type']

    # --- Build one-hot encoding pipeline ---
    indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in string_cols]
    encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe") for c in string_cols]

    pipeline = Pipeline(stages=indexers + encoders)
    df_transformed = pipeline.fit(df).transform(df)

    # --- Drop original string columns and indices ---
    cols_to_drop = string_cols + [c + "_idx" for c in string_cols]
    df_transformed = df_transformed.drop(*cols_to_drop)

    # --- Save parquet ---
    filepath = f"{output_dir}/{city_config['label']}.parquet"
    if save_parquet:
        os.makedirs(output_dir, exist_ok=True)
        df_transformed.write.mode("overwrite").parquet(filepath)
        print(f"Saved preprocessed data for {city_config['label']} → {filepath}")

    return df_transformed

In [10]:
df_preprocessed_dict = {}

# --- Run preprocessing for all cities ---
for city in CITY_CONFIGS:
    print(f"Processing {city['label']}...")
    df_preprocessed = preprocess_city(city)
    df_preprocessed_dict[city['label']] = df_preprocessed
    df_preprocessed.show(5, truncate=False)

Processing new-york-city...
Saved preprocessed data for new-york-city → data/preprocessed/new-york-city.parquet
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-----------------+--------------+--------+---------+------------+--------------+-----------------+
|id                                                                                                                                                                                                                                                             |price|number_of_reviews|minimum_nights|latitude|longitude|accommodates|room_type_ohe |property_type_ohe|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Modeling

We will do modeling per city since as seen from our EDA, there is simpson's paradox present. So doing it per city would reflect a more accurate analysis. We will also test whether certain models can predict better on each city.

In [11]:
from typing import Any, Dict, Optional, Type, Union
import numpy as np
import pandas as pd

class ForecastingMetrics:
    """Class object that contains static methods for relevant
    forecasting metrics.

    Example Usage:
    --------------
    >>> from utils import ForecastingMetrics as forecast_metrics
    >>> forecast_metrics.mae(y_true, y_pred)
    """

    @staticmethod
    def mae(y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> np.float32:
        return np.float32(np.mean(np.abs(y_true - y_pred)))

    @staticmethod
    def mse(y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> np.float32:
        return np.float32(np.mean((y_true - y_pred) ** 2))

    @staticmethod
    def rmse(y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> np.float32:
        return np.float32(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    @staticmethod
    def mape(y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> np.float32:
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        nonzero_filter = y_true != 0
        y_true = y_true[nonzero_filter]
        y_pred = y_pred[nonzero_filter]
        return np.float32(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

    @staticmethod
    def smape(y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> np.float32:
        numerator = np.abs(y_true - y_pred)
        denominator = np.abs(y_true) + np.abs(y_pred)
        nonzero_filter = denominator != 0
        return np.float32(np.mean(numerator[nonzero_filter] / (denominator[nonzero_filter] / 2)) * 100)

    @staticmethod
    def r2(y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> np.float32:
        """Compute R-squared (coefficient of determination)."""
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return np.float32(1 - ss_res / ss_tot)

    @classmethod
    def compute_all_metrics(cls, y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]) -> Dict[str, np.float32]:
        return {
            "MAE": cls.mae(y_true, y_pred),
            "MSE": cls.mse(y_true, y_pred),
            "RMSE": cls.rmse(y_true, y_pred),
            "MAPE": cls.mape(y_true, y_pred),
            "SMAPE": cls.smape(y_true, y_pred),
            "R2": cls.r2(y_true, y_pred),
        }

## Baseline: Linear Regression

In [12]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import col
import numpy as np

# Assume df_preprocessed_dict contains preprocessed DataFrames per city
# Example: df_preprocessed_dict = {"new-york-city": df_nyc, "los-angeles": df_la, ...}

train_dfs = {}
test_dfs = {}

# --- Train/Test Split per city ---
for city_name, df in df_preprocessed_dict.items():
    print(f"Splitting train/test for {city_name}...")
    train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
    train_dfs[city_name] = train_df
    test_dfs[city_name] = test_df

# --- Linear Regression ---
lr_models = {}
lr_predictions = {}

for city_name in train_dfs.keys():
    print(f"\nTraining Linear Regression for {city_name}")
    
    # --- Assemble features ---
    # Exclude target 'price' and vector columns
    numeric_features = [
        c for c in train_dfs[city_name].columns 
        if c != "price" and not c.endswith("_ohe")
    ]
    
    assembler = VectorAssembler(
        inputCols=numeric_features,
        outputCol="features"
    )
    
    train_features = assembler.transform(train_dfs[city_name]).select(
        "features", col("price").alias("label")
    )
    test_features = assembler.transform(test_dfs[city_name]).select(
        "features", col("price").alias("label")
    )
    
    # --- Train Linear Regression ---
    lr = LinearRegression(featuresCol="features", labelCol="label")
    lr_model = lr.fit(train_features)
    lr_models[city_name] = lr_model
    
    # --- Predict on test set ---
    preds = lr_model.transform(test_features)
    lr_predictions[city_name] = preds
    
    # --- Show example predictions ---
    print(f"Example predictions for {city_name}:")
    preds.select("features", "label", "prediction").show(5, truncate=False)
    
    # --- Compute metrics (using numpy arrays) ---
    y_true = np.array([row['label'] for row in preds.select("label").collect()])
    y_pred = np.array([row['prediction'] for row in preds.select("prediction").collect()])
    
    # Assuming ForecastingMetrics.compute_all_metrics exists
    metrics = ForecastingMetrics.compute_all_metrics(y_true, y_pred)
    print(metrics)

Splitting train/test for new-york-city...
Splitting train/test for los-angeles...
Splitting train/test for chicago...

Training Linear Regression for new-york-city


IllegalArgumentException: Data type string of column id is not supported.